# Week 7 Module 01 Lab: Containerizing the Cordwell Review Summarizer

**Scenario.** You are a platform engineer at Cordwell Home and Hardware. The data science
team hands you `review-summarizer`, a small service that condenses customer product reviews
into a pros-and-cons blurb. It runs fine on the author's laptop. Your job is to make it run
the same way everywhere: build it into an image, run it as a container, reconfigure it at
runtime without a rebuild, feed it data through volumes, point it at the model server on
your Mac, and finally compose it into a two-service stack behind an API gateway, with a
named volume that remembers what the containers forget.

**Time budget:** about 2.5 hours for the core parts; about 3 hours with stretch goals.
Suggested pacing is listed at the top of each part.

## Learning objectives

By the end of this lab you can:

1. Write a cache-friendly, non-root Dockerfile for a Python service and build it with BuildKit.
2. Keep the build context lean with a `.dockerignore`.
3. Run a container with published ports and readable logs, and change its behavior at
   runtime with environment overrides instead of rebuilds.
4. Mount data with bind volumes, including a read-only input, and keep outputs after
   the container exits.
5. Call a model server on the host from inside a container using `host.docker.internal`.
6. Compose a multi-service stack with service DNS and a healthcheck readiness gate.
7. Persist service state in a named volume that outlives its containers.

## Time plan

| Part | Focus | Minutes |
|---|---|---|
| 0 | Setup, environment check, meet the service | 10 |
| 1 | The Dockerfile and the build context | 30 |
| 2 | Build it with BuildKit, watch the cache | 15 |
| 3 | Run it, then reconfigure it at runtime | 25 |
| 4 | Batch job with bind volumes | 15 |
| 5 | Call the model server on the host | 15 |
| 6 | Compose the stack, cache in a named volume | 35 |
| Stretch | Multi-stage build, image healthcheck, scaling | fast finishers |

## How this lab works

- All files live in a `cordwell-lab` directory that this notebook creates next to itself.
- The service code is pre-written and already tested. Your work is the Docker side:
  the Dockerfile, the `.dockerignore`, five commands, and the Compose file.
- Each task has a check cell. Statuses mean:
  - `PASS` verified, live against Docker where possible
  - `TODO` not attempted yet
  - `FAIL` attempted, with a specific reason to fix
  - `INFO` the static check passed but Docker was unavailable for the live part
- Two hint files ship with this lab. Pick one tier per task, not both:
  `HINTS.md` (progressive nudges) or `HINTS_DETAILED.md` (working core with commentary).

**Before you start:** Docker Desktop (or equivalent) must be running, and host ports
`8000`, `8001`, and `5005` must be free. This lab avoids host port `5000` on purpose:
macOS AirPlay Receiver listens there.

If not already executed, download **python-3.13-slim.tar** from the **images** folder at <https://gamuttechnologysvcs-my.sharepoint.com/:f:/p/asanders/IgD_SIVCz8YJQYh7BL3DUy4ZAVgU8-9SO8Lo3boIy-wwV8g?e=D4X53b>. From the location where **python-3.13-slim.tar** has been stored, run `docker load -i python-3.13-slim.tar` to load the image to local cache. While you will not be creating a container from this image, you will be building a new image that use this image as a base.

> **Safe to Run All.** Every check degrades to `TODO` or a skip notice instead of
> crashing, so running the whole notebook top to bottom on a fresh copy is safe.
> Expect the scoreboard to read 0 of 8 until you start filling in tasks.

## Part 0: Setup and environment check (about 10 minutes)

Run the next four cells. They create the lab directory, load the check harness,
and confirm Docker is reachable. Nothing here requires edits.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path

LAB = Path("cordwell-lab")
for sub in ["app", "data", "out", "gateway"]:
    (LAB / sub).mkdir(parents=True, exist_ok=True)


def write_file(relpath: str, content: str) -> None:
    """Write one file into the lab directory and confirm it landed."""
    path = LAB / relpath
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print(f"wrote {path} ({len(content.splitlines())} lines)")


print(f"Lab directory ready at: {LAB.resolve()}")

In [ ]:
"""Soft-check harness for the Week 7 Docker lab.

Every check reports one of four statuses and never raises on a cold run:

  PASS  the task is verified (live against Docker when available)
  TODO  the task has not been attempted yet
  FAIL  the task was attempted but something specific is wrong
  INFO  environment notes, or a static pass when Docker is unavailable

Static checks read the files and command strings you produce. Live checks
talk to Docker and to the running containers. When Docker is not available,
static results are reported and the live portion is skipped with a note.
"""

import json
import re
import shutil
import subprocess
import time
from pathlib import Path

import requests
import yaml


def _retry_request(method: str, url: str, attempts: int = 10, delay: float = 1.5, **kwargs):
    """Containers need a moment to boot; retry the HTTP call briefly."""
    last_exc = None
    for _ in range(attempts):
        try:
            response = requests.request(method, url, timeout=10, **kwargs)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(delay)
    raise last_exc

LAB = Path("cordwell-lab")

TEST_REVIEW = "Blade arrived sharp and cuts smooth. The guard is a little flimsy though."
EXPECTED_SUMMARY = "Pros: Blade arrived sharp and cuts smooth. Cons: The guard is a little flimsy though."

IMAGE_TAG = "cordwell/review-summarizer:v1"
CONTAINER_NAME = "review-summarizer"
GATEWAY_PORT = 5005

RESULTS: dict[str, tuple[str, str]] = {}

TASK_ORDER = [
    "Task 1 Dockerfile", "Task 2 .dockerignore", "Task 3 Build", "Task 4 Run",
    "Task 5 Override", "Task 6 Batch volumes", "Task 7 Host LLM", "Task 8 Compose",
]


def _report(task: str, status: str, detail: str) -> str:
    RESULTS[task] = (status, detail)
    print(f"[{status}] {task}: {detail}")
    return status


def docker_status() -> tuple[bool, str]:
    """Is the Docker CLI present and the daemon reachable?"""
    if shutil.which("docker") is None:
        return False, "Docker CLI not found on PATH"
    probe = subprocess.run(
        ["docker", "info"], capture_output=True, text=True, timeout=30
    )
    if probe.returncode != 0:
        return False, "Docker daemon not reachable (is Docker Desktop running?)"
    return True, "Docker is available"


def run_shell(command: str, cwd: Path = LAB, timeout: int = 600):
    """Run one shell command string from the lab directory.

    Skips politely (no traceback) when the command still contains a TODO
    marker or when Docker is required but unavailable.
    """
    if "TODO" in command:
        print("Skipped: this command still contains a TODO marker. Fill it in first.")
        return None
    if command.strip().startswith("docker"):
        ok, msg = docker_status()
        if not ok:
            print(f"Skipped: {msg}.")
            return None
    print(f"$ {command}")
    proc = subprocess.run(
        command, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    print(f"(exit code {proc.returncode})")
    return proc


def _instructions(dockerfile_text: str) -> list[str]:
    """Dockerfile lines with comments and blanks stripped."""
    lines = []
    for raw in dockerfile_text.splitlines():
        line = raw.strip()
        if line and not line.startswith("#"):
            lines.append(line)
    return lines


def check_task1_dockerfile() -> str:
    path = LAB / "Dockerfile"
    if not path.exists():
        return _report("Task 1 Dockerfile", "TODO", "cordwell-lab/Dockerfile not written yet")
    lines = _instructions(path.read_text(encoding="utf-8"))
    if not lines:
        return _report("Task 1 Dockerfile", "TODO", "Dockerfile has no instructions yet")

    text = "\n".join(lines)
    from_lines = [l for l in lines if l.upper().startswith("FROM")]
    if not from_lines:
        return _report("Task 1 Dockerfile", "FAIL", "no FROM instruction found")
    if "python:3.13-slim" not in from_lines[0]:
        return _report(
            "Task 1 Dockerfile", "FAIL",
            f"base image should be pinned to python:3.13-slim, found: {from_lines[0]}",
        )
    if ":latest" in text:
        return _report("Task 1 Dockerfile", "FAIL", "never use a :latest tag; pin the version")

    def idx(pattern: str) -> int:
        for i, l in enumerate(lines):
            if re.search(pattern, l):
                return i
        return -1

    i_req = idx(r"^COPY\s+requirements\.txt")
    i_pip = idx(r"^RUN\s+.*pip\s+install")
    i_app = idx(r"^COPY\s+app/?\s")
    if i_req < 0:
        return _report("Task 1 Dockerfile", "FAIL", "missing: COPY requirements.txt .")
    if i_pip < 0:
        return _report("Task 1 Dockerfile", "FAIL", "missing: RUN pip install for requirements.txt")
    if i_app < 0:
        return _report("Task 1 Dockerfile", "FAIL", "missing: COPY app/ ./app/")
    if not (i_req < i_pip < i_app):
        return _report(
            "Task 1 Dockerfile", "FAIL",
            "layer order is wrong: requirements copy, then pip install, then app copy "
            "(cheapest-to-change last so code edits do not re-run pip install)",
        )
    if "--no-cache-dir" not in text:
        return _report("Task 1 Dockerfile", "FAIL", "add --no-cache-dir to pip install to keep the image small")

    i_useradd = idx(r"^RUN\s+useradd")
    i_user = idx(r"^USER\s+appuser")
    i_cmd = idx(r"^CMD\s")
    if i_useradd < 0 or i_user < 0:
        return _report("Task 1 Dockerfile", "FAIL", "create user appuser and switch to it (non-root)")
    if "mkdir -p /cache" not in text:
        return _report(
            "Task 1 Dockerfile", "FAIL",
            "create /cache in the image (mkdir -p /cache) so the Part 6 named volume "
            "inherits appuser ownership",
        )
    if not re.search(r"chown\s+-R\s+appuser\s+(/app\s+/cache|/cache\s+/app)", text):
        return _report(
            "Task 1 Dockerfile", "FAIL",
            "hand both directories to the user: chown -R appuser /app /cache",
        )
    if i_cmd < 0:
        return _report("Task 1 Dockerfile", "FAIL", "missing CMD to start the service")
    if not (i_useradd < i_user < i_cmd):
        return _report("Task 1 Dockerfile", "FAIL", "order should be: RUN useradd, then USER appuser, then CMD")
    if not re.search(r'^CMD\s+\["python",\s*"-m",\s*"app\.serve"\]', "\n".join(lines), re.M):
        return _report(
            "Task 1 Dockerfile", "FAIL",
            'CMD should be the exec form: CMD ["python", "-m", "app.serve"]',
        )
    if not re.search(r"^ENV\s+BACKEND_MODE=offline", text, re.M):
        return _report("Task 1 Dockerfile", "FAIL", "set a safe default: ENV BACKEND_MODE=offline")
    return _report(
        "Task 1 Dockerfile", "PASS",
        "pinned base, cache-friendly order, non-root, /cache prepared, safe defaults",
    )


def check_task2_dockerignore() -> str:
    path = LAB / ".dockerignore"
    if not path.exists():
        return _report("Task 2 .dockerignore", "TODO", "cordwell-lab/.dockerignore not written yet")
    entries = [l.strip() for l in path.read_text(encoding="utf-8").splitlines() if l.strip() and not l.startswith("#")]
    if not entries:
        return _report("Task 2 .dockerignore", "TODO", ".dockerignore is empty")
    text = "\n".join(entries)
    missing = []
    for needed, why in [
        ("data", "the reviews dataset is mounted at runtime, never baked into the image"),
        ("out", "batch outputs do not belong in the build context"),
        ("__pycache__", "compiled Python cache files"),
        (".git", "repository history bloats the context"),
    ]:
        if needed not in text:
            missing.append(f"{needed} ({why})")
    if missing:
        return _report("Task 2 .dockerignore", "FAIL", "missing entries: " + "; ".join(missing))
    return _report("Task 2 .dockerignore", "PASS", "build context excludes data, outputs, caches, and git history")


def check_task3_build(build_command: str) -> str:
    if "TODO" in build_command:
        return _report("Task 3 Build", "TODO", "BUILD_COMMAND still contains a TODO marker")
    problems = []
    if "docker build" not in build_command and "docker buildx build" not in build_command:
        problems.append("use docker build (BuildKit is the default builder)")
    if f"-t {IMAGE_TAG}" not in build_command:
        problems.append(f"tag the image with -t {IMAGE_TAG}")
    if not build_command.rstrip().endswith("."):
        problems.append("end with the build context: a single dot for the current directory")
    if problems:
        return _report("Task 3 Build", "FAIL", "; ".join(problems))
    ok, msg = docker_status()
    if not ok:
        return _report("Task 3 Build", "INFO", f"command looks right; live check skipped ({msg})")
    probe = subprocess.run(
        ["docker", "image", "inspect", IMAGE_TAG], capture_output=True, text=True
    )
    if probe.returncode != 0:
        return _report("Task 3 Build", "FAIL", f"image {IMAGE_TAG} not found; run the build cell above")
    return _report("Task 3 Build", "PASS", f"image {IMAGE_TAG} exists locally")


def check_task4_run(run_command: str) -> str:
    if "TODO" in run_command:
        return _report("Task 4 Run", "TODO", "RUN_COMMAND still contains a TODO marker")
    problems = []
    for token, why in [
        ("docker run", "start with docker run"),
        ("-d", "run detached so the notebook is not blocked"),
        ("-p 8000:8000", "publish container port 8000 to host port 8000"),
        (f"--name {CONTAINER_NAME}", f"name the container {CONTAINER_NAME}"),
        ("-e BACKEND_MODE=offline", "be explicit about the backend mode"),
        (IMAGE_TAG, f"run the image you built: {IMAGE_TAG}"),
    ]:
        if token not in run_command:
            problems.append(why)
    if problems:
        return _report("Task 4 Run", "FAIL", "; ".join(problems))
    ok, msg = docker_status()
    if not ok:
        return _report("Task 4 Run", "INFO", f"command looks right; live check skipped ({msg})")
    ps = subprocess.run(
        ["docker", "ps", "--filter", f"name={CONTAINER_NAME}", "--format", "{{.Names}} {{.Status}}"],
        capture_output=True, text=True,
    )
    if CONTAINER_NAME not in ps.stdout:
        return _report("Task 4 Run", "FAIL", "container is not running; check docker logs review-summarizer")
    try:
        health = _retry_request("GET", "http://localhost:8000/health")
        if health.get("backend_mode") != "offline":
            return _report("Task 4 Run", "FAIL", f"health reports backend_mode={health.get('backend_mode')!r}, expected offline")
        resp = _retry_request(
            "POST", "http://localhost:8000/summarize", json={"review_text": TEST_REVIEW}
        )
        if resp.get("summary") != EXPECTED_SUMMARY:
            return _report("Task 4 Run", "FAIL", f"unexpected summary: {resp.get('summary')!r}")
    except requests.RequestException as exc:
        return _report("Task 4 Run", "FAIL", f"could not reach http://localhost:8000 ({exc})")
    return _report("Task 4 Run", "PASS", "container is up and the offline summarizer answered correctly")


def check_task5_override(override_command: str) -> str:
    if "TODO" in override_command:
        return _report("Task 5 Override", "TODO", "OVERRIDE_COMMAND still contains a TODO marker")
    problems = []
    for token, why in [
        ("docker run", "start with docker run"),
        ("-d", "run detached"),
        ("-p 8000:8000", "publish the same port mapping as Task 4"),
        (f"--name {CONTAINER_NAME}", f"reuse the container name {CONTAINER_NAME}"),
        ("-e BACKEND_MODE=offline", "keep the backend offline"),
        ("-e MODEL=cordwell-eval", "override the model: -e MODEL=cordwell-eval"),
        (IMAGE_TAG, f"run the same image, no rebuild: {IMAGE_TAG}"),
    ]:
        if token not in override_command:
            problems.append(why)
    if problems:
        return _report("Task 5 Override", "FAIL", "; ".join(problems))
    ok, msg = docker_status()
    if not ok:
        return _report("Task 5 Override", "INFO", f"command looks right; live check skipped ({msg})")
    try:
        health = _retry_request("GET", "http://localhost:8000/health")
    except requests.RequestException as exc:
        return _report("Task 5 Override", "FAIL", f"could not reach http://localhost:8000 ({exc})")
    if health.get("model") != "cordwell-eval":
        return _report(
            "Task 5 Override", "FAIL",
            f"health reports model={health.get('model')!r}; the -e override did not land "
            "(did the replacement container start?)",
        )
    if health.get("backend_mode") != "offline":
        return _report("Task 5 Override", "FAIL", f"backend_mode changed to {health.get('backend_mode')!r}; keep it offline")
    return _report("Task 5 Override", "PASS", "same image reports model=cordwell-eval: config changed with zero rebuilds")


def check_task6_batch(batch_command: str) -> str:
    if "TODO" in batch_command:
        return _report("Task 6 Batch volumes", "TODO", "BATCH_COMMAND still contains a TODO marker")
    problems = []
    if "docker run" not in batch_command:
        problems.append("start with docker run")
    if "--rm" not in batch_command:
        problems.append("use --rm so the finished job cleans itself up")
    if not re.search(r"-v\s+\S*reviews\.csv:/data/reviews\.csv:ro", batch_command):
        problems.append("mount the CSV read-only: -v with :ro ending, container path /data/reviews.csv")
    if not re.search(r"-v\s+\S*/out:/out", batch_command):
        problems.append("mount an output directory at /out")
    if "python -m app.batch" not in batch_command:
        problems.append("override the command: python -m app.batch")
    if "--in /data/reviews.csv" not in batch_command or "--out /out" not in batch_command:
        problems.append("pass --in /data/reviews.csv and --out /out to app.batch")
    if problems:
        return _report("Task 6 Batch volumes", "FAIL", "; ".join(problems))
    ok, msg = docker_status()
    out_file = LAB / "out" / "summaries.json"
    if not ok:
        return _report("Task 6 Batch volumes", "INFO", f"command looks right; live check skipped ({msg})")
    if not out_file.exists():
        return _report("Task 6 Batch volumes", "FAIL", "cordwell-lab/out/summaries.json not found; run the batch cell above")
    data = json.loads(out_file.read_text(encoding="utf-8"))
    if len(data) != 12:
        return _report("Task 6 Batch volumes", "FAIL", f"expected 12 summaries, found {len(data)}")
    first = data[0]
    if first.get("review_id") != "r001" or "Pros:" not in first.get("summary", ""):
        return _report("Task 6 Batch volumes", "FAIL", f"first summary looks wrong: {first}")
    return _report("Task 6 Batch volumes", "PASS", "12 summaries written to the host through the bind mount")


def check_task7_host_llm(host_llm_command: str) -> str:
    if "TODO" in host_llm_command:
        return _report("Task 7 Host LLM", "TODO", "HOST_LLM_COMMAND still contains a TODO marker")
    mode_match = re.search(r"-e\s+BACKEND_MODE=(ollama|lmstudio)", host_llm_command)
    if not mode_match:
        return _report("Task 7 Host LLM", "FAIL", "set -e BACKEND_MODE=ollama or -e BACKEND_MODE=lmstudio")
    mode = mode_match.group(1)
    if "host.docker.internal" not in host_llm_command:
        return _report(
            "Task 7 Host LLM", "FAIL",
            "inside a container, localhost is the container itself; reach the host at host.docker.internal",
        )
    if "localhost:1234" in host_llm_command or "localhost:11434" in host_llm_command:
        return _report("Task 7 Host LLM", "FAIL", "replace localhost with host.docker.internal in the base URL")
    expected_port = "11434" if mode == "ollama" else "1234"
    if expected_port not in host_llm_command:
        return _report(
            "Task 7 Host LLM", "FAIL",
            f"backend {mode} listens on port {expected_port}; the base URL should use it",
        )
    return _report(
        "Task 7 Host LLM", "PASS",
        f"routes to {mode} on the host via host.docker.internal:{expected_port} "
        "(a live model server is optional for this task)",
    )


def check_task8_compose() -> str:
    path = LAB / "compose.yaml"
    if not path.exists():
        return _report("Task 8 Compose", "TODO", "cordwell-lab/compose.yaml not written yet")
    raw = path.read_text(encoding="utf-8")
    if "TODO" in raw:
        return _report("Task 8 Compose", "TODO", "compose.yaml still contains TODO markers")
    try:
        doc = yaml.safe_load(raw)
    except yaml.YAMLError as exc:
        return _report("Task 8 Compose", "FAIL", f"compose.yaml is not valid YAML: {exc}")
    services = (doc or {}).get("services") or {}
    if "summarizer" not in services or "gateway" not in services:
        return _report("Task 8 Compose", "FAIL", "define both services: summarizer and gateway")
    summ, gate = services["summarizer"], services["gateway"]
    if "ports" in summ:
        return _report("Task 8 Compose", "FAIL", "summarizer must stay private: remove its ports key")
    env_text = json.dumps(summ.get("environment") or [])
    if "CACHE_DIR" not in env_text or "/cache" not in env_text:
        return _report("Task 8 Compose", "FAIL", "summarizer needs CACHE_DIR=/cache so it caches into the volume")
    vols = [str(v) for v in (summ.get("volumes") or [])]
    if not any(v.startswith("summary_cache:") and "/cache" in v for v in vols):
        return _report("Task 8 Compose", "FAIL", "mount the named volume: summary_cache:/cache under the summarizer")
    top_vols = (doc or {}).get("volumes") or {}
    if "summary_cache" not in top_vols:
        return _report("Task 8 Compose", "FAIL", "declare the named volume in a top-level volumes: section")
    hc = summ.get("healthcheck") or {}
    if not hc.get("test"):
        return _report("Task 8 Compose", "FAIL", "summarizer needs a healthcheck with a test command")
    ports = [str(p) for p in (gate.get("ports") or [])]
    if f"{GATEWAY_PORT}:8000" not in ports:
        return _report("Task 8 Compose", "FAIL", f'gateway should publish "{GATEWAY_PORT}:8000" (port 5000 collides with AirPlay on macOS)')
    dep = gate.get("depends_on")
    cond = None
    if isinstance(dep, dict):
        cond = (dep.get("summarizer") or {}).get("condition")
    if cond != "service_healthy":
        return _report(
            "Task 8 Compose", "FAIL",
            "gateway must gate on readiness: depends_on summarizer with condition service_healthy",
        )
    env = gate.get("environment") or []
    env_text = json.dumps(env)
    if "http://summarizer:8000" not in env_text:
        return _report(
            "Task 8 Compose", "FAIL",
            "gateway needs SUMMARIZER_URL=http://summarizer:8000 (Compose DNS by service name, not an IP)",
        )
    ok, msg = docker_status()
    if not ok:
        return _report("Task 8 Compose", "INFO", f"file looks right; live check skipped ({msg})")
    ps = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"], capture_output=True, text=True, cwd=LAB
    )
    rows = []
    for line in ps.stdout.strip().splitlines():
        try:
            item = json.loads(line)
            rows.extend(item if isinstance(item, list) else [item])
        except json.JSONDecodeError:
            pass
    running = {r.get("Service"): r.get("State") for r in rows}
    if running.get("summarizer") != "running" or running.get("gateway") != "running":
        return _report("Task 8 Compose", "FAIL", f"stack is not fully running yet: {running or 'no services found'}")
    try:
        first = _retry_request(
            "POST", f"http://localhost:{GATEWAY_PORT}/summarize",
            json={"review_text": TEST_REVIEW},
        )
        if first.get("via") != "gateway" or first.get("summary") != EXPECTED_SUMMARY:
            return _report("Task 8 Compose", "FAIL", f"gateway responded, but payload is wrong: {first}")
        second = _retry_request(
            "POST", f"http://localhost:{GATEWAY_PORT}/summarize",
            json={"review_text": TEST_REVIEW},
        )
        if second.get("cached") is not True:
            return _report(
                "Task 8 Compose", "FAIL",
                "second identical request was not a cache hit; is CACHE_DIR=/cache set "
                "and the summary_cache volume mounted?",
            )
    except requests.RequestException as exc:
        return _report("Task 8 Compose", "FAIL", f"could not reach the gateway on port {GATEWAY_PORT} ({exc})")
    return _report(
        "Task 8 Compose", "PASS",
        "stack healthy: request flowed host to gateway to summarizer by DNS name, "
        "and the repeat request hit the named-volume cache",
    )


def scoreboard() -> None:
    print("\n===== Lab scoreboard =====")
    for task in TASK_ORDER:
        status, detail = RESULTS.get(task, ("TODO", "not checked yet"))
        print(f"[{status}] {task}: {detail}")
    passed = sum(1 for t in TASK_ORDER if RESULTS.get(t, ("", ""))[0] == "PASS")
    print(f"\n{passed} of {len(TASK_ORDER)} tasks fully verified.")
    if any(RESULTS.get(t, ("", ""))[0] == "INFO" for t in TASK_ORDER):
        print("INFO means the static check passed but Docker was unavailable for live verification.")

In [ ]:
ok, msg = docker_status()
print(msg)
run_shell("docker version --format 'client={{.Client.Version}} server={{.Server.Version}}'")
run_shell("docker compose version")

## Part 1: The service you are shipping, and its Dockerfile (about 30 minutes)

First, meet the code. Three small modules, all pre-written and pre-tested, so your
attention stays on Docker:

- `app/backend.py` holds the `BACKEND_MODE` selector you have used all course:
  `offline` (default, deterministic, no server needed), `lmstudio` (host port `1234`),
  and `ollama` (host port `11434`). The two live backends share one OpenAI-compatible
  code path and differ only by base URL. The model name lives in the `MODEL` config
  variable, defaulting to `gemma4`.
- `app/serve.py` is the HTTP surface: `GET /health`, `GET /cache-stats`, and
  `POST /summarize`. When the `CACHE_DIR` environment variable is set, it stores each
  summary under a hash of its input and answers identical requests from that cache,
  reporting `"cached": true`. With `CACHE_DIR` unset, caching is off. This matters in
  Part 6, where a named volume becomes the cache.
- `app/batch.py` reads a CSV of reviews and writes one JSON file of summaries.
  You will use it in the volumes part.

One detail worth noticing: the base URLs default to `host.docker.internal`, not
`localhost`. This code expects to live inside a container, and inside a container
`localhost` means the container itself. More on that in Part 5.

Run the next cells to write the code, the pinned dependency list, and the seeded
synthetic reviews dataset into `cordwell-lab`.

In [ ]:
write_file("app/__init__.py", r"""""")

In [ ]:
write_file("app/backend.py", r'''"""Backend selector for the Cordwell review summarizer.

Three modes, selected by the BACKEND_MODE environment variable:

  offline   (default) deterministic rule-based summarizer, no server needed
  lmstudio  OpenAI-compatible server on the host, port 1234
  ollama    OpenAI-compatible server on the host, port 11434

The two live backends share one code path because both speak the
OpenAI-compatible chat completions API. Only the base URL differs.
Inside a container, "the host" is reached at host.docker.internal,
so that is the default hostname here. An unknown mode raises a clear
ValueError instead of silently falling back.
"""

import os

import requests

# Config lives in variables, never hard-coded at call sites.
BACKEND_MODE = os.getenv("BACKEND_MODE", "offline").strip().lower()
MODEL = os.getenv("MODEL", "gemma4")

# Defaults assume this code runs inside a container talking to the host.
# Override with env vars when running directly on the host machine.
LMSTUDIO_BASE_URL = os.getenv(
    "LMSTUDIO_BASE_URL", "http://host.docker.internal:1234/v1"
)
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL", "http://host.docker.internal:11434"
)

# Small keyword lexicons for the deterministic offline mode.
_POSITIVE = {
    "great", "love", "loved", "excellent", "perfect", "sturdy", "easy",
    "solid", "fast", "quiet", "smooth", "durable", "reliable", "sharp",
    "comfortable", "powerful", "accurate", "best", "happy", "recommend",
}
_NEGATIVE = {
    "broke", "broken", "flimsy", "cheap", "loud", "slow", "leaks", "leaked",
    "rust", "rusted", "difficult", "hard", "confusing", "disappointed",
    "return", "returned", "worst", "wobbly", "stripped", "died", "defective",
}


def _split_sentences(text: str) -> list[str]:
    """Very small sentence splitter: good enough for short reviews."""
    parts = []
    current = []
    for ch in text:
        current.append(ch)
        if ch in ".!?":
            parts.append("".join(current).strip())
            current = []
    tail = "".join(current).strip()
    if tail:
        parts.append(tail)
    return [p for p in parts if p]


def summarize_offline(review_text: str) -> str:
    """Deterministic pros-and-cons blurb. Same input, same output, always."""
    pros: list[str] = []
    cons: list[str] = []
    for sentence in _split_sentences(review_text):
        words = {w.strip(".,!?;:()").lower() for w in sentence.split()}
        pos_hits = len(words & _POSITIVE)
        neg_hits = len(words & _NEGATIVE)
        if pos_hits > neg_hits:
            pros.append(sentence)
        elif neg_hits > pos_hits:
            cons.append(sentence)
    pros_part = " ".join(pros) if pros else "None noted."
    cons_part = " ".join(cons) if cons else "None noted."
    return f"Pros: {pros_part} Cons: {cons_part}"


def _chat_completion(base_url: str, review_text: str) -> str:
    """One shared OpenAI-compatible call path for both live backends."""
    url = base_url.rstrip("/") + "/chat/completions"
    prompt = (
        "Summarize this Cordwell customer product review as a one-line "
        "pros-and-cons blurb: " + review_text
    )
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
    }
    response = requests.post(url, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]


def summarize(review_text: str) -> str:
    """Route the request based on BACKEND_MODE."""
    if BACKEND_MODE == "offline":
        return summarize_offline(review_text)
    if BACKEND_MODE == "lmstudio":
        return _chat_completion(LMSTUDIO_BASE_URL, review_text)
    if BACKEND_MODE == "ollama":
        # Ollama exposes the OpenAI-compatible API under /v1.
        return _chat_completion(OLLAMA_BASE_URL.rstrip("/") + "/v1", review_text)
    raise ValueError(
        f"Unknown BACKEND_MODE={BACKEND_MODE!r}. "
        "Use one of: offline, lmstudio, ollama."
    )
''')

In [ ]:
write_file("app/serve.py", r'''"""HTTP surface for the Cordwell review summarizer.

Endpoints:
  GET  /health       liveness probe used by Docker healthchecks
  GET  /cache-stats  how many summaries are cached (when CACHE_DIR is set)
  POST /summarize    body {"review_text": "..."} returns {"summary": "...", ...}

Caching: when the CACHE_DIR environment variable is set (the Compose stack
points it at a named volume), each summary is stored under a hash of its
input text. An identical request is then answered from the cache, and the
response reports "cached": true. With CACHE_DIR unset, nothing is cached
and "cached" is always false.

Run inside the container with:
  python -m app.serve
"""

import hashlib
import json
import os
from pathlib import Path

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from app import backend

app = FastAPI(title="Cordwell Review Summarizer", version="2.0")


class SummarizeRequest(BaseModel):
    review_text: str


def _cache_dir() -> Path | None:
    raw = os.getenv("CACHE_DIR")
    return Path(raw) if raw else None


def _cache_key(review_text: str) -> str:
    return hashlib.sha256(review_text.encode("utf-8")).hexdigest()


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "backend_mode": backend.BACKEND_MODE, "model": backend.MODEL}


@app.get("/cache-stats")
def cache_stats() -> dict:
    cache = _cache_dir()
    if cache is None:
        return {"cache_enabled": False, "cached_summaries": 0}
    count = len(list(cache.glob("*.json"))) if cache.exists() else 0
    return {"cache_enabled": True, "cached_summaries": count}


@app.post("/summarize")
def summarize(req: SummarizeRequest) -> dict:
    text = req.review_text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="review_text must be non-empty")

    cache = _cache_dir()
    key = _cache_key(text)
    if cache is not None:
        cached_path = cache / f"{key}.json"
        if cached_path.exists():
            stored = json.loads(cached_path.read_text(encoding="utf-8"))
            stored["cached"] = True
            return stored

    try:
        summary = backend.summarize(text)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:  # live backend unreachable, bad response, etc.
        raise HTTPException(
            status_code=502,
            detail=f"Backend {backend.BACKEND_MODE!r} call failed: {exc}",
        ) from exc

    result = {
        "summary": summary,
        "backend_mode": backend.BACKEND_MODE,
        "model": backend.MODEL,
    }
    if cache is not None:
        cache.mkdir(parents=True, exist_ok=True)
        (cache / f"{key}.json").write_text(json.dumps(result), encoding="utf-8")
    result["cached"] = False
    return result


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=8000)
''')

In [ ]:
write_file("app/batch.py", r'''"""Batch mode: summarize a CSV of reviews and write one JSON file of results.

Used in the volumes exercise. The input CSV is mounted read-only, and the
output directory is a bind mount so results survive after the container exits.

  python -m app.batch --in /data/reviews.csv --out /out
"""

import argparse
import csv
import json
from pathlib import Path

from app import backend


def main() -> None:
    parser = argparse.ArgumentParser(description="Cordwell batch review summarizer")
    parser.add_argument("--in", dest="in_path", required=True, help="input CSV path")
    parser.add_argument("--out", dest="out_dir", required=True, help="output directory")
    args = parser.parse_args()

    in_path = Path(args.in_path)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    results = []
    with in_path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            results.append(
                {
                    "review_id": row["review_id"],
                    "sku": row["sku"],
                    "summary": backend.summarize(row["review_text"]),
                }
            )

    out_path = out_dir / "summaries.json"
    out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(f"Wrote {len(results)} summaries to {out_path} (backend={backend.BACKEND_MODE})")


if __name__ == "__main__":
    main()
''')

In [ ]:
write_file("requirements.txt", r"""fastapi==0.141.1
uvicorn==0.52.3
requests==2.34.2
pydantic==2.13.4
""")

In [ ]:
write_file("data/reviews.csv", r"""review_id,sku,product_name,review_text
r001,CW-DRL-018,Cordless Drill 18V,This drill is sturdy and the battery lasts all weekend. The chuck feels solid. Love it.
r002,CW-SAW-204,Circular Saw 7.25in,Blade arrived sharp and cuts smooth. The guard is a little flimsy though.
r003,CW-LMP-330,Clamp Work Lamp,Great lamp for the workbench. Bright and easy to position. The clamp is wobbly on thick tables.
r004,CW-HOS-112,Garden Hose 50ft,Hose leaks at the coupling after two weeks. Returned it. Disappointed.
r005,CW-FAN-450,Shop Fan 20in,"Quiet fan, moves a lot of air. Assembly was easy. Best shop fan I have owned."
r006,CW-SHV-077,Steel Shelving Unit,Shelving unit was easy to assemble and feels durable. One bracket arrived slightly bent.
r007,CW-FAN-450,Shop Fan 20in,Motor died after a month. Cheap internals. Worst purchase this year.
r008,CW-DRL-018,Cordless Drill 18V,Powerful and reliable. I recommend this to anyone finishing a basement.
r009,CW-SHV-077,Steel Shelving Unit,Instructions were confusing and the screws stripped easily. Hard to recommend.
r010,CW-LMP-330,Clamp Work Lamp,"Excellent value. Solid build, fast shipping, and it runs quiet."
r011,CW-HOS-112,Garden Hose 50ft,The finish rusted after one rainy weekend outside. Flimsy coating.
r012,CW-SAW-204,Circular Saw 7.25in,Comfortable grip and accurate cuts. My go-to saw now. Happy with it.
""")

### Test before you ship

A container is a shipping decision, not a debugging tool. Prove the code works on
your machine first, so that any failure after this point is a Docker problem, not
a Python problem. The cell below calls the offline summarizer directly, no Docker
involved. This exact input and output pair is also what the later checks verify.

In [ ]:
import subprocess
import sys

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import sys; sys.path.insert(0, '.'); "
        "from app.backend import summarize_offline; "
        "print(summarize_offline('Blade arrived sharp and cuts smooth. "
        "The guard is a little flimsy though.'))",
    ],
    cwd=LAB,
    capture_output=True,
    text=True,
)
print(probe.stdout or probe.stderr)

### Task 1: Write the Dockerfile

Time to package it. Edit the `DOCKERFILE` string in the next cell, then run it to
write the file. The cell after that checks your work.

**Contract** (the check verifies each of these):

1. Base image pinned to `python:3.13-slim`. Never `:latest`. The cohort runtime is Python 3.13.
2. Working directory is `/app`.
3. Dependencies are installed before the application code is copied in, so a code
   edit does not re-run `pip install`. Copy the dependency list, install it with pip,
   and pass `--no-cache-dir` so the pip download cache never lands in a layer.
4. Copy the `app` package into the image after the install step.
5. Create a user named `appuser` with a home directory, create the `/cache` directory,
   hand both `/app` and `/cache` to that user, and switch to it. Root inside a
   container is a bad default. Why `/cache` exists now when nothing uses it yet is a
   Part 6 payoff; one `RUN` line covers all three actions.
6. Bake safe defaults into the image: `BACKEND_MODE=offline` and `MODEL=gemma4`.
   These are plain config, not secrets, so `ENV` is the right tool. Override at
   runtime with `-e`, no rebuild needed.
7. Document the service port with `EXPOSE 8000`, and start the service with the
   exec-form command: `CMD ["python", "-m", "app.serve"]`

**Worked target output.** When your Dockerfile is right, the check cell prints:

```
[PASS] Task 1 Dockerfile: pinned base, cache-friendly order, non-root, /cache prepared, safe defaults
```

In [ ]:
DOCKERFILE = r"""# TODO Task 1: build this Dockerfile to the contract above.
# Replace these comments with real instructions; the check cell verifies
# each numbered requirement and tells you exactly what is missing.
"""

write_file("Dockerfile", DOCKERFILE)

In [ ]:
check_task1_dockerfile()

### Task 2: Keep the build context lean

When you run `docker build .`, the trailing dot is the build context: everything in
the directory gets sent to the Docker daemon before the build starts. Datasets,
outputs, caches, and git history all ride along unless you say otherwise, and a
bloated context makes every build slow. The reviews dataset is mounted at runtime
in this lab, so it must never be baked into the image or shipped in the context.

Edit the `DOCKERIGNORE` string so the context excludes at least: the `data`
directory, the `out` directory, `__pycache__` directories, and the `.git` directory.
Excluding local virtual environments and large model files is good practice too.

**Worked target output.**

```
[PASS] Task 2 .dockerignore: build context excludes data, outputs, caches, and git history
```

In [ ]:
DOCKERIGNORE = r"""# TODO Task 2: one exclusion per line, per the contract above.
"""

write_file(".dockerignore", DOCKERIGNORE)

In [ ]:
check_task2_dockerignore()

## Part 2: Build it with BuildKit (about 15 minutes)

### Task 3: The build command

`docker build` is an alias for `docker buildx build` now: BuildKit is the default
builder, which gives you parallel layer resolution and better caching for free.

Fill in `BUILD_COMMAND` below. It should tag the image `cordwell/review-summarizer:v1`
and use the current directory as the build context. The `run_shell` helper executes
the command from inside `cordwell-lab`, so the current directory is the right context.

**Worked target output.** A successful first build ends with lines shaped like this
(hashes will differ, timing will differ):

```
 => exporting to image
 => => naming to docker.io/cordwell/review-summarizer:v1
```

The first build downloads the base image and installs dependencies, so expect
roughly 1 to 3 minutes. That is normal, not broken.

In [ ]:
BUILD_COMMAND = "TODO: the docker build command per the contract above"

run_shell(BUILD_COMMAND, timeout=900)

In [ ]:
check_task3_build(BUILD_COMMAND)

### Watch the cache earn its keep

Run the same build a second time. Nothing changed, so every step should say
`CACHED` and the build finishes in about a second.

Then the cell after that appends one comment line to `app/serve.py` and rebuilds.
Watch the output closely: the pip install layer stays `CACHED`, and only the
`COPY app/` step onward re-runs. That is the payoff of putting dependencies before
code. If you had copied the code first, this one-line edit would re-run the whole
`pip install`.

In [ ]:
run_shell(BUILD_COMMAND)

In [ ]:
serve_path = LAB / "app" / "serve.py"
marker = "# cache demo: one-line code edit"
text = serve_path.read_text(encoding="utf-8")
if marker not in text:
    serve_path.write_text(text + "\n" + marker + "\n", encoding="utf-8")
    print("Appended one comment line to app/serve.py")
else:
    print("Marker already present; rebuilding anyway")
run_shell(BUILD_COMMAND)

## Part 3: Run it, then reconfigure it at runtime (about 25 minutes)

### Task 4: The run command

Fill in `RUN_COMMAND`. The contract:

- detached (`-d`) so the notebook is not blocked
- publish container port `8000` on host port `8000` (`-p host:container`)
- name the container `review-summarizer` so later commands can refer to it
- set `-e BACKEND_MODE=offline` explicitly, even though it is the image default,
  because being explicit about config at run time is the habit that saves you later
- run the image you built: `cordwell/review-summarizer:v1`

**Worked target output.** Once the container is up, the verification cell prints
exactly this:

```
GET /health -> {'status': 'ok', 'backend_mode': 'offline', 'model': 'gemma4'}
POST /summarize -> Pros: Blade arrived sharp and cuts smooth. Cons: The guard is a little flimsy though.
```

In [ ]:
RUN_COMMAND = "TODO: the docker run command per the contract above"

run_shell(RUN_COMMAND)

In [ ]:
ok, msg = docker_status()
if "TODO" in RUN_COMMAND:
    print("Fill in RUN_COMMAND above first.")
elif not ok:
    print(f"Skipped: {msg}")
else:
    try:
        health = _retry_request("GET", "http://localhost:8000/health")
        print(f"GET /health -> {health}")
        resp = _retry_request(
            "POST", "http://localhost:8000/summarize", json={"review_text": TEST_REVIEW}
        )
        print(f"POST /summarize -> {resp['summary']}")
    except Exception as exc:
        print(f"Could not reach the service yet: {exc}")
        print("Check the container state with the logs cell below.")

In [ ]:
check_task4_run(RUN_COMMAND)

In [ ]:
run_shell("docker ps --filter name=review-summarizer")
run_shell("docker logs --tail 20 review-summarizer")

### Task 5: Override configuration at runtime

Here is the whole point of `ENV` defaults plus `-e` overrides: **change behavior
without rebuilding the image**. You will replace the running container with one
whose `MODEL` is overridden to `cordwell-eval`. Offline mode never calls a model,
which is exactly why this is safe to demonstrate: the lesson is watching config
flow from the command line into the running process, not model output.

Container names must be unique, so the provided line in the next cell removes the
Task 4 container first. Fill in `OVERRIDE_COMMAND`:

- same shape as Task 4: detached, `-p 8000:8000`, `--name review-summarizer`,
  the same `v1` image (no rebuild)
- keep `-e BACKEND_MODE=offline`
- override the model: `-e MODEL=cordwell-eval`

**Worked target output.** The verification cell prints:

```
GET /health -> {'status': 'ok', 'backend_mode': 'offline', 'model': 'cordwell-eval'}
```

Same image, different behavior, zero rebuilds.

In [ ]:
run_shell("docker rm -f review-summarizer")

OVERRIDE_COMMAND = "TODO: the docker run command with the MODEL override, per the contract above"

run_shell(OVERRIDE_COMMAND)

In [ ]:
ok, msg = docker_status()
if "TODO" in OVERRIDE_COMMAND:
    print("Fill in OVERRIDE_COMMAND above first.")
elif not ok:
    print(f"Skipped: {msg}")
else:
    try:
        health = _retry_request("GET", "http://localhost:8000/health")
        print(f"GET /health -> {health}")
    except Exception as exc:
        print(f"Could not reach the service yet: {exc}")

In [ ]:
check_task5_override(OVERRIDE_COMMAND)

## Part 4: Volumes, because containers forget (about 15 minutes)

Stop and think about what the last two containers had in common: everything they
wrote lives in a writable layer that vanishes when the container is removed. For a
batch job that produces results you want to keep, that is a problem. Bind mounts
solve it by mapping host paths into the container.

### Task 6: The batch command

Fill in `BATCH_COMMAND` to run the batch summarizer as a one-off job:

- `docker run --rm` so the finished job removes itself
- mount the input read-only: host `data/reviews.csv` to container `/data/reviews.csv`
  with a `:ro` suffix. Read-only protects source data: even buggy code cannot
  corrupt the input.
- mount an output directory: host `out` to container `/out`
- keep `-e BACKEND_MODE=offline`
- override the image command: `python -m app.batch --in /data/reviews.csv --out /out`

Use `"$(pwd)/data/reviews.csv"` and `"$(pwd)/out"` for the host paths. Bind mounts
need absolute paths, and `run_shell` executes from inside `cordwell-lab`, so
`$(pwd)` expands to the right place.

**Worked target output.** The job prints one line and exits, and the first entry
of `cordwell-lab/out/summaries.json`, read from the host after the container is
gone, looks exactly like this:

```
Wrote 12 summaries to /out/summaries.json (backend=offline)
```

```json
{
  "review_id": "r001",
  "sku": "CW-DRL-018",
  "summary": "Pros: This drill is sturdy and the battery lasts all weekend. The chuck feels solid. Love it. Cons: None noted."
}
```

In [ ]:
BATCH_COMMAND = "TODO: the docker run command for the batch job, per the contract above"

run_shell(BATCH_COMMAND)

In [ ]:
import json

out_file = LAB / "out" / "summaries.json"
if out_file.exists():
    data = json.loads(out_file.read_text(encoding="utf-8"))
    print(f"{len(data)} summaries survive on the host after the container exited:\n")
    print(json.dumps(data[:2], indent=2))
else:
    print("No output yet: run the batch cell above first.")

In [ ]:
check_task6_batch(BATCH_COMMAND)

### Who wrote those files?

One habit from Task 1 quietly paid off here. Your image drops root before the
process starts, so the batch job ran as `appuser`, not root. The `id` output below
proves it.

Now look at the ownership of the results with `ls -ln`. On your Mac the files
appear owned by your own uid, because Docker Desktop's file sharing mediates
ownership between the Linux VM and macOS. On a plain Linux host the container's
uid shows through directly, and the mirror-image problem bites: a **root**
container writes root-owned files into your bind mount that your own user then
cannot clean up, and a **non-root** container can fail to write into a root-owned
mount at all. The standard fixes are matching uids, or handing writable state to
named volumes, which is exactly what Part 6 does with the cache. File the Linux
case away for Week 8, when these containers leave your laptop.

In [ ]:
run_shell(f"docker run --rm {IMAGE_TAG} id")
if (LAB / "out").exists() and any((LAB / "out").iterdir()):
    run_shell("ls -ln out")
else:
    print("No out/ directory contents yet: run the Task 6 batch job first.")

## Part 5: Call the model server on the host (about 15 minutes)

Here is the trap this part exists to spring. On your Mac, Ollama listens at
`http://localhost:11434` and LM Studio at `http://localhost:1234`. Inside a
container, those URLs point at the container itself, and the call fails with a
connection refused. The container and your Mac are different network namespaces
with different meanings of `localhost`.

Docker Desktop provides a stable DNS name for the host: `host.docker.internal`.
Swap it in and the same container reaches the model server running natively on
your Mac, which is where Apple Silicon acceleration lives. No code changes, only
environment variables: that is the whole point of the `BACKEND_MODE` pattern.

### Task 7: The host-backend run command

Fill in `HOST_LLM_COMMAND` for a container that routes to a host model server:

- pick a backend: `-e BACKEND_MODE=ollama` or `-e BACKEND_MODE=lmstudio`
- set the matching base URL env var through `host.docker.internal`:
  `OLLAMA_BASE_URL=http://host.docker.internal:11434` for Ollama, or
  `LMSTUDIO_BASE_URL=http://host.docker.internal:1234/v1` for LM Studio
- set `-e MODEL=gemma4` (confirm the exact tag your machine has pulled)
- publish a different host port so it does not collide with the running Task 5
  container: `-p 8001:8000`
- add `--rm` and `--name review-summarizer-live` so cleanup stays easy, plus `-d`

The check validates the command shape. Actually calling a live model is optional:
if you have Ollama or LM Studio running, flip `TRY_LIVE_BACKEND = True` two cells
down to see a real model answer through the container.

In [ ]:
HOST_LLM_COMMAND = "TODO: the docker run command for the host-backend container, per the contract above"

In [ ]:
check_task7_host_llm(HOST_LLM_COMMAND)

In [ ]:
TRY_LIVE_BACKEND = False  # set True if Ollama or LM Studio is running on your Mac
ok, msg = docker_status()
if TRY_LIVE_BACKEND and "TODO" in HOST_LLM_COMMAND:
    print("Fill in HOST_LLM_COMMAND above first.")
elif TRY_LIVE_BACKEND and not ok:
    print(f"Skipped: {msg}")
elif TRY_LIVE_BACKEND:
    run_shell(HOST_LLM_COMMAND)
    try:
        health = _retry_request("GET", "http://localhost:8001/health", attempts=5)
        print(f"GET /health -> {health}")
        resp = _retry_request(
            "POST", "http://localhost:8001/summarize",
            json={"review_text": TEST_REVIEW}, attempts=3,
        )
        print(f"Live model summary -> {resp['summary']}")
    except Exception as exc:
        print(f"Live backend not reachable: {exc}")
        print("Is the model server running on the host? Explicit selection fails loudly on purpose.")
    finally:
        run_shell("docker rm -f review-summarizer-live")
else:
    print("Skipped live backend test. Set TRY_LIVE_BACKEND = True to try it.")

## Part 6: Compose the Cordwell stack (about 35 minutes)

Real systems are several services. Yours will be two:

- `summarizer`: the image you built, kept private to the Compose network. No ports.
- `gateway`: a small pre-written proxy, the only service published to the host,
  on port `5005`. It reaches the summarizer by DNS name, `http://summarizer:8000`,
  because Compose gives every service a DNS entry matching its service key.

The request flow you are wiring: your Mac calls `localhost:5005`, the gateway
forwards to `summarizer:8000` over the private Compose network, the summarizer
answers, and the gateway stamps the response with `"via": "gateway"`.

Why port `5005` and not `5000` like the architecture diagrams? On macOS, AirPlay
Receiver squats on port `5000`. Fighting the operating system for a port is not a
lesson worth 20 minutes of class time.

Two concepts before you write YAML:

**Readiness, not just start order.** `depends_on` alone only controls start
order. It waits for the summarizer container to start, not for the service inside
it to be ready. A healthcheck plus `condition: service_healthy` is the readiness
gate: Compose polls the healthcheck command and holds the gateway back until the
summarizer actually answers.

**Named volumes, because this stack remembers.** The summarizer caches summaries
under `/cache` when `CACHE_DIR` points there, and the stack mounts a named volume
called `summary_cache` at that path. Unlike the bind mounts of Part 4, a named
volume is storage Docker owns and manages by name, and `docker compose down`
removes containers and the network but keeps named volumes. Here is the payoff of
a Task 1 detail you may have wondered about: the Dockerfile created `/cache` and
handed it to `appuser`. When Docker first mounts an empty named volume at a path
that exists in the image, the volume inherits that path's ownership. Without that
line, the volume would mount root-owned and your non-root service could not write
a single cache entry.

First, clean up the Part 3 container, then write the gateway files.

In [ ]:
run_shell("docker rm -f review-summarizer")

In [ ]:
write_file("gateway/serve.py", r'''"""Cordwell API gateway.

The only service published to the host in the Compose stack. It forwards
summarize requests to the summarizer service by its Compose DNS name.
SUMMARIZER_URL is injected by Compose, so this code never hard-codes an IP.
"""

import os

import requests
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

SUMMARIZER_URL = os.getenv("SUMMARIZER_URL", "http://summarizer:8000")

app = FastAPI(title="Cordwell API Gateway", version="1.0")


class SummarizeRequest(BaseModel):
    review_text: str


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "summarizer_url": SUMMARIZER_URL}


@app.post("/summarize")
def summarize(req: SummarizeRequest) -> dict:
    try:
        response = requests.post(
            SUMMARIZER_URL.rstrip("/") + "/summarize",
            json={"review_text": req.review_text},
            timeout=60,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        raise HTTPException(
            status_code=502, detail=f"summarizer call failed: {exc}"
        ) from exc
    payload = response.json()
    payload["via"] = "gateway"
    return payload
''')

In [ ]:
write_file("gateway/requirements.txt", r"""fastapi==0.141.1
uvicorn==0.52.3
requests==2.34.2
pydantic==2.13.4
""")

In [ ]:
write_file("gateway/Dockerfile", r"""FROM python:3.13-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY serve.py .

RUN useradd -m appuser && chown -R appuser /app
USER appuser

EXPOSE 8000
CMD ["python", "-m", "uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "8000"]
""")

### Task 8: Complete the Compose file

Fill in the `COMPOSE_YAML` skeleton. The contract:

- `summarizer`: builds from `.`, sets `BACKEND_MODE=offline` and `CACHE_DIR=/cache`,
  mounts the named volume `summary_cache` at `/cache`, has a healthcheck, and has
  no `ports` key. The slim image has no curl, so the healthcheck test uses python
  and urllib (the skeleton gives you the exact line). Use `interval: 5s`,
  `timeout: 3s`, `retries: 5`, `start_period: 10s`.
- `gateway`: builds from `./gateway`, publishes `"5005:8000"`, sets
  `SUMMARIZER_URL=http://summarizer:8000`, and depends on the summarizer with
  `condition: service_healthy`.
- a top-level `volumes:` section declaring `summary_cache`. Compose V2 needs no
  top-level `version:` key; it ignores one and warns.

**Worked target output.** After `docker compose up -d --build`, `docker compose ps`
shows both services running with the summarizer healthy, and the check cell makes
two requests through the gateway. The first misses the cache, the second hits it:

```
[PASS] Task 8 Compose: stack healthy: request flowed host to gateway to summarizer by DNS name, and the repeat request hit the named-volume cache
```

In [ ]:
COMPOSE_YAML = r"""services:
  summarizer:
    build: .
    environment:
      - BACKEND_MODE=offline
      - MODEL=gemma4
      # TODO 1: point the cache at the volume mount path
    # TODO 2: mount the named volume summary_cache at /cache
    # No ports key here on purpose: the summarizer stays private.
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health').read()"]
      # TODO 3: the four timing keys from the contract
    # Portable mapping so host.docker.internal also resolves on Linux hosts.
    extra_hosts:
      - "host.docker.internal:host-gateway"

  gateway:
    build: ./gateway
    # TODO 4: publish the gateway per the contract
    environment:
      - SUMMARIZER_URL=http://summarizer:8000
    # TODO 5: depends_on with the readiness condition

# TODO 6: declare the top-level named volume
"""

write_file("compose.yaml", COMPOSE_YAML)

In [ ]:
run_shell("docker compose up -d --build", timeout=900)

In [ ]:
run_shell("docker compose ps")

In [ ]:
check_task8_compose()

### Prove the DNS claim

Exec into the running gateway container and call the summarizer by its service
name. No IP addresses anywhere. This is the private network the diagram promised.

In [ ]:
run_shell(
    "docker compose exec -T gateway python -c "
    "\"import requests; print(requests.get('http://summarizer:8000/health', timeout=5).json())\""
)

### The cache at work, and the closed front door

Two identical requests through the gateway. The first computes the summary and
stores it in the named volume; the second is answered from the cache. Then one
more claim to verify: the summarizer publishes no ports, so from the host it
should not answer at all. Only the gateway's door is open.

In [ ]:
ok, msg = docker_status()
compose_path = LAB / "compose.yaml"
compose_ready = compose_path.exists() and "TODO" not in compose_path.read_text(encoding="utf-8")
if not ok:
    print(f"Skipped: {msg}")
elif not compose_ready:
    print("Finish Task 8 first: compose.yaml still has TODO markers.")
else:
    try:
        payload = {"review_text": TEST_REVIEW}
        first = _retry_request("POST", f"http://localhost:{GATEWAY_PORT}/summarize", json=payload)
        second = _retry_request("POST", f"http://localhost:{GATEWAY_PORT}/summarize", json=payload)
        print(f"first call:  cached={first['cached']}  via={first['via']}")
        print(f"second call: cached={second['cached']}  via={second['via']}")
    except Exception as exc:
        print(f"Gateway not reachable: {exc}")
    print()
    print("Straight at the summarizer from the host (this should NOT answer):")
    try:
        requests.get("http://localhost:8000/health", timeout=3)
        print("Unexpected: the summarizer answered on the host. Look for a stray Part 3 container.")
    except requests.RequestException:
        print("Connection refused, as designed. Only the gateway is published.")

### The memorable part: the volume outlives the stack

`docker compose down` removes the containers and the network. It does **not**
remove named volumes. Tear the whole stack down, bring it back, repeat the
request: it is served from cache by a container that did not exist a minute ago.
Containers forget; volumes remember. The `cache-stats` call at the end, made
inside the fresh summarizer container, counts the entries that survived.

In [ ]:
ok, msg = docker_status()
compose_path = LAB / "compose.yaml"
compose_ready = compose_path.exists() and "TODO" not in compose_path.read_text(encoding="utf-8")
if not ok:
    print(f"Skipped: {msg}")
elif not compose_ready:
    print("Finish Task 8 first: compose.yaml still has TODO markers.")
else:
    run_shell("docker compose down")
    print("Stack is gone: containers and network removed, named volume kept.\n")
    run_shell("docker compose up -d")
    try:
        resp = _retry_request(
            "POST", f"http://localhost:{GATEWAY_PORT}/summarize",
            json={"review_text": TEST_REVIEW}, attempts=20,
        )
        print(f"cached={resp['cached']} from brand-new containers: that summary lived in the named volume.")
    except Exception as exc:
        print(f"Gateway did not come back: {exc}")
    run_shell(
        "docker compose exec -T summarizer python -c "
        "\"import urllib.request; print(urllib.request.urlopen('http://localhost:8000/cache-stats').read().decode())\""
    )

## Stretch goals (optional, if time allows)

**Stretch 1: Multi-stage build.** Rewrite the Dockerfile as a two-stage build:
a `builder` stage that creates a virtual environment at `/venv` and installs the
dependencies into it, and a final `python:3.13-slim` stage that copies only the
venv and the app code, sets `PATH` so the venv python wins, and keeps the non-root
and safe-default conventions (including `/cache`). Save it as
`Dockerfile.multistage`, build it with
`docker build -f Dockerfile.multistage -t cordwell/review-summarizer:v2 .`, and
compare image sizes with `docker images cordwell/review-summarizer`. Ship the
artifact, not the toolchain.

**Stretch 2: Bake a healthcheck into the image.** Compose supplied the
healthcheck in Task 8; an image can also carry its own. Add a `HEALTHCHECK`
instruction to a copy of your Dockerfile (exec form, the same `python -c` urllib
probe, since slim has no curl), save it as `Dockerfile.health`, and build
`cordwell/review-summarizer:health`. Run it detached on host port `8001` and
watch `docker ps` flip from `(health: starting)` to `(healthy)`. Done when
`docker inspect --format '{{.State.Health.Status}}'` prints `healthy`. Bonus
question: if the image has a `HEALTHCHECK` and the Compose file also defines one
for that service, which wins?

**Stretch 3: Scaling and the port gotcha.** Scale the private service:
`docker compose up -d --scale summarizer=3`. Confirm with `docker compose ps` and
make several requests through the gateway. Then try
`docker compose up -d --scale gateway=2` and explain the error you get. Why does
one scale cleanly while the other fails, and what are the two standard fixes?

Work from the contracts above. `HINTS_DETAILED.md` covers the stretch goals at
the same depth as the core tasks; fully assembled stretch solutions live in the
instructor notebook.

## Cleanup

Set `DO_CLEANUP = True` and run the cell when you are done exploring. It tears
down the Compose stack and removes the standalone containers, and it leaves two
things in place deliberately: the images, so a re-run of this notebook is fast,
and the `summary_cache` named volume, because `docker compose down` never removes
named volumes. That survival is the Part 6 lesson; when you truly want the volume
gone, that is what `docker compose down -v` is for.

In [ ]:
DO_CLEANUP = False  # set True to tear everything down

if DO_CLEANUP:
    run_shell("docker compose down")
    run_shell("docker rm -f review-summarizer review-summarizer-live review-health")
    print("Containers and the Compose network are gone. The named volume remains:")
    run_shell("docker volume ls --filter name=summary_cache")
    print()
    print("To also remove the volume:  docker compose down -v")
    print("To remove the images:       docker rmi cordwell/review-summarizer:v1 "
          "cordwell/review-summarizer:v2 cordwell/review-summarizer:health")
else:
    print("Skipped. Set DO_CLEANUP = True to tear down the containers.")

## Wrap-up: talk it through

Discuss these with your table before we regroup:

1. The same image ran offline, ran with a runtime model override, and routed to a
   host Ollama, all without one code change. What made that possible, and where
   else in a Cordwell service would you apply the same pattern?
2. `depends_on` versus `condition: service_healthy`: describe a concrete failure
   you would see in production if the gateway used the plain list form and the
   summarizer took 20 seconds to load a model.
3. The summarizer had no `ports:` entry at all. What attack surface did that
   remove, and what would you publish in a staging environment where you want to
   poke the summarizer directly?
4. `docker compose down` kept your cache; `docker compose down -v` would not.
   Name one kind of state at Cordwell you would want in a named volume and one
   kind you would refuse to keep there, and defend both choices.

Keep the project directory. Module 02 picks up the same image for pipeline
versioning, and Week 8 pushes it toward a registry and CI.